# Lensless Imaging Demo Bonus

demo for bonus, a trained non-admm solver (unrolled fista-20) and a pretrained general-purpose gan restoration network fine-tuned on top of admm-100. we clone the repo, install dependencies, download the checkpoints and a demo dataset, reconstruct the measurements, visualize the results and compute metrics

### Cloning repo

In [ ]:
import os

if not os.path.exists('src'):
    !git clone https://github.com/arslan-yam/lensless_imaging
    %cd lensless_imaging

### Installing dependencies and imports

In [ ]:
!pip install -q -r requirements.txt

import gdown
import matplotlib.pyplot as plt
from pathlib import Path
from src.datasets.custom_dir import IMAGE_EXTS, read_image
from lensless_helpers.preprocessor import get_roi

## Download weights and dataset

fista_unrolled and admm100_sr_ft are the trained model_best.pth checkpoints; dataset is a zipped demo set that must unzip to data/custom_test/ with lensless/, masks/, and lensed/ subfolders

In [ ]:
os.makedirs("saved/fista_unrolled", exist_ok=True)
os.makedirs("saved/admm100_sr_ft", exist_ok=True)
gdown.download(id="1L8EYTaAvKuOw3QB1WN0TOIhP52KrFSpi", output="saved/fista_unrolled/model_best.pth", quiet=False)
gdown.download(id="1c1VBP2zzZ5F3V1vSgs5vetIRCJJJckvU", output="saved/admm100_sr_ft/model_best.pth", quiet=False)

dataset = "1nVQ_Lbdz2XCMdzM5l8s-RY_Ui3hWTfs4" #replace with your link
gdown.download(id=dataset, output="custom_test.zip", quiet=False)
!mkdir -p data
!unzip -q -o custom_test.zip -d data/
!ls data/custom_test

## Reconstruction

reconstruct the demo maesurements with two methods. each run writes the reconstructed png files to data/saved/<save_path>/test/.

unrolled fista

In [ ]:
!python inference.py model=fista_unrolled inferencer.from_pretrained=saved/fista_unrolled/model_best.pth datasets.test.path=data/custom_test inferencer.save_path=fista_unrolled

admm 100 + real-esrgan (fine-tuned)

In [ ]:
!python inference.py model=admm100_sr_ft inferencer.from_pretrained=saved/admm100_sr_ft/model_best.pth datasets.test.path=data/custom_test inferencer.save_path=admm100_sr_ft

## Visualize reconstructions

for some examples, show the lensless measurement, unrolled fista, admm 100 + real-esrgan fine-tuned, and the ground truth.

In [ ]:
data_dir = Path("data/custom_test")

def find_image(directory, stem):
    directory = Path(directory)
    for ext in IMAGE_EXTS:
        path = directory / f"{stem}{ext}"
        if path.exists():
            return path
    raise FileNotFoundError(f"No image found for {stem} in {directory}")

def read_required(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Image file not found: {path}")
    try:
        return read_image(path)
    except AttributeError as exc:
        raise ValueError(f"OpenCV cannot read image: {path}") from exc

names = sorted(p.stem for p in (data_dir / "lensless").iterdir() if p.suffix.lower() in IMAGE_EXTS)

def show(name):
    lensless = read_required(find_image(data_dir / "lensless", name))
    lensed = read_required(find_image(data_dir / "lensed", name))
    fista = get_roi(read_required(find_image("data/saved/fista_unrolled/test", name)))
    sr_ft = get_roi(read_required(find_image("data/saved/admm100_sr_ft/test", name)))
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    for ax, img, title in zip(axes, [lensless, fista, sr_ft, lensed], ["lensless measurement", "unrolled fista-20", "admm-100 + real-esrgan (ft)", "ground truth"]):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis("off")
    plt.show()

for name in names[:3]:
    show(name)

## Compute metrics

calculate_metrics.py compares the saved reconstructions against the ground-truth lensed images.

In [ ]:
print("unrolled fista-20:")
!python calculate_metrics.py --reconstructions data/saved/fista_unrolled/test --dataset data/custom_test

print("admm-100 + real-esrgan (fine-tuned):")
!python calculate_metrics.py --reconstructions data/saved/admm100_sr_ft/test --dataset data/custom_test